# 4 · How realistic is this market?

Realism here is a stated envelope rather than a score. Ten statistics are
measured against real-market bands, and the gaps are named, including one
that is structurally unreachable.

In [1]:
import pretium as pt
from pretium import envelope as env
from pretium.facts import REAL_MARKETS, band_distance

print("preset certified :", env.PRESET)
print("certified horizon:", env.CERTIFIED_HORIZON_DAYS, "trading days")

preset certified : pt-v10
certified horizon: 252 trading days


## The panel

Measured at 30 seeds, 40 instruments, 252 days.

In [2]:
print(f"{'statistic':26s} {'measured':>10s} {'band':>20s}   verdict")
for name in REAL_MARKETS:
    lo, hi = REAL_MARKETS[name]
    v = env.CERTIFIED[name]
    ok = band_distance(v, lo, hi) == 0
    print(f"{name:26s} {v:10.4f} {f'[{lo:g}, {hi:g}]':>20s}   "
          f"{'in band' if ok else 'OUT'}")

n = sum(1 for k in REAL_MARKETS
        if band_distance(env.CERTIFIED[k], *REAL_MARKETS[k]) == 0)
print(f"\n{n} of {len(REAL_MARKETS)} in band")

statistic                    measured                 band   verdict
annualised_vol_pct            31.4632             [15, 36]   in band
excess_kurtosis                7.7618            [1.6, 41]   in band
return_acf1                    0.0195        [-0.08, 0.06]   in band
abs_return_acf1                0.0994         [0.02, 0.22]   in band
abs_return_acf5                0.0487         [0.02, 0.09]   in band
abs_return_acf20               0.0043        [-0.04, 0.08]   in band
cross_sectional_corr           0.3063         [0.08, 0.56]   in band
volume_abs_return_corr         0.4784         [0.46, 0.66]   in band
leverage_effect               -0.0336           [-0.16, 0]   in band
volume_change_acf1            -0.3130        [-0.32, -0.2]   in band
corr_asymmetry                -0.0034        [-0.25, 0.45]   in band
corr_asymmetry_lagged          0.0054         [-0.2, 0.55]   in band
sector_excess_corr             0.1346         [0.11, 0.23]   in band
corr_persistence_acf1          0.1

## Nothing fails at one year, and one thing fails at two

Every statistic is inside its band at the certified horizon. That is new
in 0.2.0: the previous default missed two, and `volume_change_acf1` was
described here as unreachable.

It is still the weak one. Its two-year band is tighter than its one-year
band, and the model sits outside it there, which is why the cell below
reads 14 of 14 against the 252-day ruler and fewer against the 504-day
one. A strategy trading the change in volume is on solid ground at one
year and outside the envelope at two.

## Scoring a panel

`envelope.score` reads a panel against the ruler for its own horizon. The
same numbers score differently at 252 and 504 days, which is the mistake
this function exists to prevent.


In [3]:
panel = {k: env.CERTIFIED[k] for k in REAL_MARKETS}

near = env.score(panel, horizon_days=252)
far = env.score(panel, horizon_days=504)

print(f"252-day ruler ({near['ruler']}): {near['in_band']}/{near['of']} in band")
print(f"504-day ruler ({far['ruler']}): {far['in_band']}/{far['of']} in band")
print()
k = "excess_kurtosis"
for label, s in (("252", near), ("504", far)):
    r = s["statistics"][k]
    print(f"  {k} @{label}d: {r['measured']:.3f} vs band {r['band']}"
          f"  -> {'in' if r['in_band'] else 'OUT'}")

252-day ruler (REAL_MARKETS): 14/14 in band
504-day ruler (REAL_MARKETS_504): 11/14 in band

  excess_kurtosis @252d: 7.762 vs band (1.6, 41.0)  -> in
  excess_kurtosis @504d: 7.762 vs band (7.1, 22.0)  -> in


`room_sd` gives the distance inside a band in that horizon's own seed noise.
A statistic barely inside is one seed away from being outside, and a plain
band check cannot distinguish the two.

In [4]:
rows = [(k, r["room_sd"]) for k, r in near["statistics"].items()
        if r["in_band"] and r["room_sd"] is not None]
for k, room in sorted(rows, key=lambda kv: kv[1]):
    flag = "  <- thin" if room < 0.5 else ""
    print(f"  {k:26s} {room:6.2f} sd inside{flag}")

  volume_abs_return_corr       0.42 sd inside  <- thin
  leverage_effect              0.44 sd inside  <- thin
  abs_return_acf5              0.52 sd inside
  volume_change_acf1           0.68 sd inside
  annualised_vol_pct           0.70 sd inside
  return_acf1                  0.76 sd inside
  abs_return_acf1              0.84 sd inside
  abs_return_acf20             0.95 sd inside
  corr_persistence_acf1        1.26 sd inside
  corr_asymmetry               1.53 sd inside
  corr_asymmetry_lagged        1.74 sd inside
  cross_sectional_corr         2.08 sd inside
  sector_excess_corr           3.62 sd inside
  excess_kurtosis              5.26 sd inside


## The gaps

Each gap names what it stops you concluding.

In [5]:
for gap in env.GAPS:
    print(f"* {gap.id}")
    print(f"    forbids: {gap.forbids}")

* volume-change
    forbids: strategies trading the day-to-day CHANGE in volume
* horizon
    forbids: multi-year backtests, and anything keyed on volatility dynamics beyond one year
* decay-shape
    forbids: strategies whose edge depends on volatility memory beyond about lag 20 -- vol targeting and risk parity on a one-month or longer estimate
* thin-tails
    forbids: tail-risk or VaR calibration at multi-year horizons
* scenario-magnitude
    forbids: sizing a scenario's impact rather than detecting it
* macro-range
    forbids: studying inflation regimes or policy crises from the endogenous economy alone
* sector-structure
    forbids: sector rotation, sector-neutral construction, industry diversification, and any long/short pair whose thesis is the sector
* roster-concentration
    forbids: inheriting this envelope for a sector-concentrated roster -- re-measure the panel on your own universe instead


## Checking your own question

`check` refuses questions that fall outside the envelope, and every refusal
names the measurement behind it.

In [6]:
questions = [
    ("a one-year momentum study", dict(horizon_days=252,
                                       statistics=["return_acf1"])),
    ("a three-year study",        dict(horizon_days=756,
                                       statistics=["abs_return_acf1"])),
    ("volatility clustering decay", dict(horizon_days=252,
                                         statistics=["abs_return_acf20"])),
    ("a tech-only roster",        dict(horizon_days=252,
                                       sector_concentrated=True)),
]

for label, kwargs in questions:
    v = env.check(**kwargs)
    print(f"{label:32s} {'INSIDE' if v.inside else 'outside'}")
    if not v.inside:
        for reason in v.reasons:
            print(f"      {reason[:96]}")

a one-year momentum study        INSIDE
a three-year study               outside
      horizon 756d exceeds the certified 252d. At 504 days the model holds 13 of 14 against horizon-ma
volatility clustering decay      outside
      abs_return_acf20 depends on the decay shape, which is a mechanism gap: log-log slope -0.953 agai
a tech-only roster               outside
      the roster is sector-concentrated, and certification was measured on a perfectly balanced one. A


## Choosing a preset

`pt-v3` is the default and what the envelope certifies at 252 days. `pt-v4`
trades some of that for multi-year behaviour: it is the first preset to
bring `excess_kurtosis` inside its 504-day band, and gives up `return_acf1`
at 252 days to do it.

`envelope.regressions` reports what a candidate would surrender.

In [7]:
import statistics as _stats
from pretium import facts

small = pt.Universe.random(40, seed=111)

def panel_for(preset, seeds=(1, 2, 3)):
    model = pt.ModelParams.from_preset(preset)
    panels = [facts.measure(seed=s, universe=small, days=252, model=model)
              for s in seeds]
    return {k: _stats.median(p[k] for p in panels) for k in REAL_MARKETS}

for preset in ("pt-v3", "pt-v4"):
    p = panel_for(preset)
    s = env.score(p)
    print(f"{preset}:  {s['in_band']}/{s['of']} in band at 252d"
          f"   regressions vs shipped: {env.regressions(p) or 'none'}")

pt-v3:  12/14 in band at 252d   regressions vs shipped: none


pt-v4:  10/14 in band at 252d   regressions vs shipped: ['return_acf1']


Three seeds is a small sample; the published figures use thirty. Treat the
counts above as indicative. What matters is the shape of the trade, and
that `regressions` reports it rather than leaving you to count by hand.

**pt-v3 for horizons at or under a year; pt-v4 for multi-year questions.**

## Summary

- Realism is a set of measurements against bands, with the failures named.
- The certified horizon is 252 days, and `check` refuses beyond it.
- Good results here do not predict real returns.

Full documentation: <https://simoncoombes.github.io/pretium/>